# 🚇 Buildings within 5-Minute Walking Distance of Berlin Transit Stations

This notebook uses **osmnx** (included in city2graph's environment) to:
1. Download the **walking street network** for a Berlin neighbourhood
2. Find all **S-Bahn, U-Bahn, and Tram stops**
3. Compute **5-minute walking isochrones** from each stop
4. Download **building footprints**
5. **Mark and visualise** which buildings fall within walking distance

> ⚠️ Running this for all of Berlin at once would be very slow and may hit API limits.  
> We use **Mitte** as a default example. Change `PLACE` to any Berlin neighbourhood.

## 0 · Configuration — change these to explore different areas

In [ ]:
# ── SETTINGS ──────────────────────────────────────────────────────────────────
PLACE          = "Mitte, Berlin, Germany"   # neighbourhood or borough
WALK_SPEED_KMH = 4.5                        # average walking speed km/h
WALK_MINUTES   = 5                          # isochrone size in minutes
# ──────────────────────────────────────────────────────────────────────────────

WALK_METRES = (WALK_SPEED_KMH * 1000 / 60) * WALK_MINUTES
print(f"Walk distance used: {WALK_METRES:.0f} m  ({WALK_MINUTES} min @ {WALK_SPEED_KMH} km/h)")

## 1 · Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import osmnx as ox
import geopandas as gpd
import pandas as pd
import networkx as nx
import folium
from shapely.geometry import Point, MultiPolygon
from shapely.ops import unary_union

print("✅ Imports OK")
print(f"   osmnx   {ox.__version__}")
print(f"   geopandas {gpd.__version__}")

## 2 · Download the walking street network

In [ ]:
print(f"Downloading walking network for: {PLACE} …")
G = ox.graph_from_place(PLACE, network_type="walk", simplify=True)
G = ox.add_edge_speeds(G)          # adds speed_kph attribute
G = ox.add_edge_travel_times(G)    # adds travel_time in seconds

nodes, edges = ox.graph_to_gdfs(G)
print(f"✅ Network: {len(nodes):,} nodes, {len(edges):,} edges")

## 3 · Download transit stations (S-Bahn, U-Bahn, Tram)

In [ ]:
print("Downloading transit stops …")

# OSM tags for Berlin rapid transit stops
transit_tags = {
    "station": ["subway", "light_rail"],   # U-Bahn + S-Bahn stations
    "railway": ["tram_stop", "station"],   # Tram stops + rail stations
    "public_transport": "stop_position"    # generic stop positions
}

transit_gdf = ox.features_from_place(PLACE, tags=transit_tags)

# Keep only point geometries (actual stops, not building polygons)
transit_stops = transit_gdf[transit_gdf.geometry.geom_type == "Point"].copy()

# Also convert polygon centroids (some stations are mapped as areas)
transit_polys = transit_gdf[transit_gdf.geometry.geom_type != "Point"].copy()
if len(transit_polys) > 0:
    transit_polys = transit_polys.copy()
    transit_polys['geometry'] = transit_polys.geometry.centroid
    transit_stops = pd.concat([transit_stops, transit_polys])

transit_stops = transit_stops.reset_index(drop=True)
transit_stops = transit_stops.to_crs(epsg=4326)

print(f"✅ Found {len(transit_stops)} transit stops")

# Show a sample
cols = [c for c in ['name','station','railway','public_transport'] if c in transit_stops.columns]
transit_stops[cols].dropna(how='all').head(10)

## 4 · Compute 5-minute walking isochrones

In [ ]:
print(f"Computing {WALK_MINUTES}-minute walking isochrones for each stop …")

trip_time_seconds = WALK_MINUTES * 60
isochrone_polys = []
skipped = 0

for idx, stop in transit_stops.iterrows():
    try:
        # Find nearest node on walking network
        nearest_node = ox.distance.nearest_nodes(
            G, stop.geometry.x, stop.geometry.y
        )
        # Ego graph: all nodes reachable within travel_time seconds
        subgraph = nx.ego_graph(
            G, nearest_node, radius=trip_time_seconds,
            distance="travel_time"
        )
        # Build convex hull from reachable node positions
        node_points = [
            Point(data["x"], data["y"])
            for _, data in subgraph.nodes(data=True)
        ]
        if len(node_points) >= 3:
            poly = gpd.GeoSeries(node_points).unary_union.convex_hull
            isochrone_polys.append(poly)
    except Exception:
        skipped += 1

print(f"✅ Computed {len(isochrone_polys)} isochrones ({skipped} stops skipped)")

# Merge all isochrones into one unified walkable zone
walkable_zone = unary_union(isochrone_polys)
walkable_gdf = gpd.GeoDataFrame(geometry=[walkable_zone], crs="EPSG:4326")
print(f"   Merged into single walkable zone")

## 5 · Download building footprints

In [ ]:
print("Downloading building footprints …")

buildings = ox.features_from_place(PLACE, tags={"building": True})
buildings = buildings[buildings.geometry.geom_type.isin(["Polygon","MultiPolygon"])].copy()
buildings = buildings.to_crs(epsg=4326).reset_index(drop=True)

print(f"✅ Found {len(buildings):,} buildings")

## 6 · Mark buildings within walking distance

In [ ]:
print("Classifying buildings …")

# Use building centroid for the intersection test (faster than full polygon)
building_centroids = buildings.geometry.centroid
buildings["within_walk"] = building_centroids.within(walkable_zone)

n_within = buildings["within_walk"].sum()
n_total  = len(buildings)
pct      = 100 * n_within / n_total if n_total > 0 else 0

print(f"✅ {n_within:,} of {n_total:,} buildings ({pct:.1f}%) are within {WALK_MINUTES} min walk of a transit stop")

## 7 · Interactive map

In [ ]:
print("Building interactive map …")

# Centre map on the area
centre = nodes.geometry.unary_union.centroid
m = folium.Map(
    location=[centre.y, centre.x],
    zoom_start=14,
    tiles="CartoDB dark_matter"
)

# ── Walkable zone (semi-transparent blue fill) ────────────────────────────────
folium.GeoJson(
    walkable_gdf.__geo_interface__,
    name="5-min walkable zone",
    style_function=lambda _: {
        "fillColor": "#4a90d9",
        "color": "#4a90d9",
        "weight": 1,
        "fillOpacity": 0.15
    }
).add_to(m)

# ── Buildings NOT within walk (grey) ─────────────────────────────────────────
far_buildings = buildings[~buildings["within_walk"]]
folium.GeoJson(
    far_buildings.__geo_interface__,
    name=f"Buildings > {WALK_MINUTES} min from transit",
    style_function=lambda _: {
        "fillColor": "#555555",
        "color": "#333333",
        "weight": 0.3,
        "fillOpacity": 0.6
    }
).add_to(m)

# ── Buildings WITHIN walk (green/yellow) ─────────────────────────────────────
near_buildings = buildings[buildings["within_walk"]]
folium.GeoJson(
    near_buildings.__geo_interface__,
    name=f"Buildings ≤ {WALK_MINUTES} min from transit",
    style_function=lambda _: {
        "fillColor": "#7ecf6e",
        "color": "#4aa83a",
        "weight": 0.5,
        "fillOpacity": 0.8
    }
).add_to(m)

# ── Transit stops (red circles) ───────────────────────────────────────────────
name_col = "name" if "name" in transit_stops.columns else None
for _, stop in transit_stops.iterrows():
    label = stop[name_col] if name_col and pd.notna(stop[name_col]) else "Transit Stop"
    folium.CircleMarker(
        location=[stop.geometry.y, stop.geometry.x],
        radius=5,
        color="#ff4444",
        fill=True,
        fill_color="#ff4444",
        fill_opacity=0.9,
        tooltip=label
    ).add_to(m)

# ── Layer control + title ─────────────────────────────────────────────────────
folium.LayerControl().add_to(m)

title_html = f'''
<div style="position:fixed; top:12px; left:60px; z-index:1000;
     background:rgba(0,0,0,0.75); color:white; padding:10px 16px;
     border-radius:6px; font-family:sans-serif; font-size:13px;">
  <b>🚇 {PLACE}</b><br>
  <span style="color:#7ecf6e">■</span> Within {WALK_MINUTES} min walk of S/U/Tram &nbsp;
  <span style="color:#888">■</span> Further away &nbsp;
  <span style="color:#ff4444">●</span> Transit stop
</div>'''
m.get_root().html.add_child(folium.Element(title_html))

# Save and display
map_path = "berlin_transit_walkability.html"
m.save(map_path)
print(f"✅ Map saved to: {map_path}")
m   # display inline in JupyterLab

## 8 · Summary statistics

In [ ]:
print("=" * 50)
print(f"  Area analysed : {PLACE}")
print(f"  Walk threshold: {WALK_MINUTES} min ({WALK_METRES:.0f} m)")
print(f"  Transit stops : {len(transit_stops)}")
print(f"  Total buildings: {n_total:,}")
print(f"  Within walk   : {n_within:,}  ({pct:.1f}%)")
print(f"  Outside walk  : {n_total - n_within:,}  ({100-pct:.1f}%)")
print("=" * 50)

---
## 💡 Tips

- **Change the area**: Edit `PLACE` in cell 0 to any Berlin borough, e.g. `"Neukölln, Berlin, Germany"`
- **Change walk time**: Edit `WALK_MINUTES` to `10` for a 10-minute isochrone
- **Larger area**: Use `"Berlin, Germany"` but expect it to take 5–15 minutes to run
- **Save the map**: The HTML map is saved in your JupyterLab file browser as `berlin_transit_walkability.html` — right-click → Download